# text

> RoBERTa text tower, as `ClapTextModel` runs it at inference.
> 
> Dropout is the identity. Position ids follow RoBERTa: padding stays at the
> pad id, and real tokens are numbered from `pad_id + 1`. The pooler is the
> first token, a linear layer, and tanh.


In [ ]:
#| default_exp text


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import numpy as np
from max.dtype import DType
from max.experimental import functional as F
from max.experimental.nn import Embedding, LayerNorm, Linear, Module, ModuleList
from max.experimental.tensor import Tensor

from nada.config import TextConfig


In [ ]:
#| export
def l2_normalize(x: Tensor) -> Tensor:
    """Divide each row by its Euclidean norm."""
    inverse = F.rsqrt(F.sum(x * x, axis=-1))
    return x * inverse.reshape([x.shape[0], 1])


In [ ]:
#| exporti
def _additive_mask(attention_mask: Tensor) -> Tensor:
    """`(batch, 1, 1, seq)` mask. Padded keys receive a large negative score."""
    negative = F.constant(
        np.float32(-1.0e4),
        dtype=DType.float32,
        device=attention_mask.device,
    )
    additive = (1.0 - attention_mask) * negative
    return additive.reshape([attention_mask.shape[0], 1, 1, attention_mask.shape[1]])


In [ ]:
#| export
def roberta_position_ids(input_ids: np.ndarray, pad_token_id: int = 1) -> np.ndarray:
    """Position ids RoBERTa assigns, including the pad id on padded steps."""
    mask = (input_ids != pad_token_id).astype(np.int64)
    return (np.cumsum(mask, axis=1) * mask) + pad_token_id


In [ ]:
#| export
class ClapProjection(Module):
    """Two linear layers with a ReLU between them. Both towers use this head."""

    def __init__(self, hidden_size: int, projection_dim: int) -> None:
        self.linear1 = Linear(hidden_size, projection_dim)
        self.linear2 = Linear(projection_dim, projection_dim)

    def forward(self, hidden: Tensor) -> Tensor:
        return self.linear2(F.relu(self.linear1(hidden)))


In [ ]:
#| export
class ClapTextEmbeddings(Module):
    """Word, position, and token-type embeddings, then LayerNorm."""

    def __init__(self, config: TextConfig) -> None:
        self.pad_token_id = config.pad_token_id
        self.hidden_size = config.hidden_size
        self.word_embeddings = Embedding(config.vocab_size, dim=config.hidden_size)
        self.position_embeddings = Embedding(config.max_position_embeddings, dim=config.hidden_size)
        self.token_type_embeddings = Embedding(config.type_vocab_size, dim=config.hidden_size)
        self.LayerNorm = LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        # Embedding draws a random table. Zero it so a model can be built
        # before the checkpoint arrives without touching that much entropy.
        self.word_embeddings.weight = Tensor.zeros(
            [config.vocab_size, config.hidden_size], dtype=DType.float32
        )
        self.position_embeddings.weight = Tensor.zeros(
            [config.max_position_embeddings, config.hidden_size], dtype=DType.float32
        )
        self.token_type_embeddings.weight = Tensor.zeros(
            [config.type_vocab_size, config.hidden_size], dtype=DType.float32
        )

    def forward(self, input_ids: Tensor) -> Tensor:
        pad = F.constant(self.pad_token_id, dtype=DType.int64, device=input_ids.device)
        valid = F.cast(F.not_equal(input_ids, pad), DType.int64)
        position_ids = F.cumsum(valid, axis=1) * valid + pad
        token_type = self.token_type_embeddings.weight.reshape([1, 1, self.hidden_size])
        hidden = self.word_embeddings(input_ids)
        hidden = hidden + self.position_embeddings(position_ids) + token_type
        return self.LayerNorm(hidden)


In [ ]:
#| export
class ClapTextSelfAttention(Module):
    """Bidirectional attention. The mask is added to the scores."""

    def __init__(self, config: TextConfig) -> None:
        if config.hidden_size % config.num_attention_heads:
            raise ValueError(f"{config.hidden_size} is not divisible by {config.num_attention_heads} heads")
        self.heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.scale = self.head_dim ** -0.5
        self.query = Linear(config.hidden_size, config.hidden_size)
        self.key = Linear(config.hidden_size, config.hidden_size)
        self.value = Linear(config.hidden_size, config.hidden_size)

    def forward(self, hidden: Tensor, mask: Tensor) -> Tensor:
        batch, seq, _ = hidden.shape[0], hidden.shape[1], hidden.shape[2]
        query = self._heads(self.query(hidden), batch, seq)
        key = self._heads(self.key(hidden), batch, seq)
        value = self._heads(self.value(hidden), batch, seq)
        scores = (query @ key.permute([0, 1, 3, 2])) * self.scale + mask
        context = F.softmax(scores, axis=-1) @ value
        return context.permute([0, 2, 1, 3]).reshape([batch, seq, self.heads * self.head_dim])

    def _heads(self, states: Tensor, batch, seq) -> Tensor:
        return states.reshape([batch, seq, self.heads, self.head_dim]).permute([0, 2, 1, 3])


In [ ]:
#| export
class ClapTextSelfOutput(Module):
    """Attention projection, residual, and LayerNorm."""

    def __init__(self, config: TextConfig) -> None:
        self.dense = Linear(config.hidden_size, config.hidden_size)
        self.LayerNorm = LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(self, hidden: Tensor, residual: Tensor) -> Tensor:
        return self.LayerNorm(self.dense(hidden) + residual)


In [ ]:
#| export
class ClapTextAttention(Module):
    def __init__(self, config: TextConfig) -> None:
        self.self = ClapTextSelfAttention(config)
        self.output = ClapTextSelfOutput(config)

    def forward(self, hidden: Tensor, mask: Tensor) -> Tensor:
        return self.output(self.self(hidden, mask), hidden)


In [ ]:
#| export
class ClapTextIntermediate(Module):
    def __init__(self, config: TextConfig) -> None:
        self.dense = Linear(config.hidden_size, config.intermediate_size)

    def forward(self, hidden: Tensor) -> Tensor:
        return F.gelu(self.dense(hidden))


In [ ]:
#| export
class ClapTextOutput(Module):
    def __init__(self, config: TextConfig) -> None:
        self.dense = Linear(config.intermediate_size, config.hidden_size)
        self.LayerNorm = LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(self, hidden: Tensor, residual: Tensor) -> Tensor:
        return self.LayerNorm(self.dense(hidden) + residual)


In [ ]:
#| export
class ClapTextLayer(Module):
    """One RoBERTa block: attention, then a GELU feed-forward."""

    def __init__(self, config: TextConfig) -> None:
        self.attention = ClapTextAttention(config)
        self.intermediate = ClapTextIntermediate(config)
        self.output = ClapTextOutput(config)

    def forward(self, hidden: Tensor, mask: Tensor) -> Tensor:
        hidden = self.attention(hidden, mask)
        return self.output(self.intermediate(hidden), hidden)


In [ ]:
#| export
class ClapTextPooler(Module):
    """Linear and tanh on the first token."""

    def __init__(self, config: TextConfig) -> None:
        self.dense = Linear(config.hidden_size, config.hidden_size)

    def forward(self, hidden: Tensor) -> Tensor:
        first = F.slice_tensor(hidden, [slice(None), 0, slice(None)])
        return F.tanh(self.dense(first))


In [ ]:
#| export
class ClapTextEncoder(Module):
    def __init__(self, config: TextConfig) -> None:
        self.layer = ModuleList([ClapTextLayer(config) for _ in range(config.num_hidden_layers)])

    def forward(self, hidden: Tensor, mask: Tensor) -> Tensor:
        for layer in self.layer:
            hidden = layer(hidden, mask)
        return hidden


In [ ]:
#| export
class ClapTextModel(Module):
    """Token ids in, pooled first-token state out."""

    def __init__(self, config: TextConfig) -> None:
        self.embeddings = ClapTextEmbeddings(config)
        self.encoder = ClapTextEncoder(config)
        self.pooler = ClapTextPooler(config)

    def forward(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        mask = _additive_mask(attention_mask)
        hidden = self.encoder(self.embeddings(input_ids), mask)
        return self.pooler(hidden)
